# FloPyArcade – Pre-Prepared Solutions

This notebook demonstrates **pre-prepared solutions** for various FloPyArcade environment types.

Each cell loads a known-optimal action sequence (the “solution”) and replays it through the environment,
showing the resulting reward and trajectory.


In [ ]:
import sys
import os

# Ensure the package is importable
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('__file__')), '..'))

import numpy as np
from flopyarcade.play import PlayEnv


## Solution Format

A **solution** is a list of action indices (integers) representing one action per timestep.
For a `ngames`-step environment the solution list has length `ngames`.

| Action | Meaning |
|--------|---------|
| 0      | Keep (no well change) |
| 1      | Move up |
| 2      | Move right |
| 3      | Move down |
| 4      | Move left |


In [ ]:
# Pre-prepared solutions keyed by environment type
SOLUTIONS = {
    '1s-d': [0, 1, 2, 0, 3, 0, 2, 0, 1, 0],
    '2s-d': [0, 2, 0, 1, 0, 3, 0, 2, 0, 1, 0, 3, 0, 2, 0],
    '3s-d': [0, 1, 0, 2, 0, 3, 0, 1, 0, 2, 0, 3, 0, 1, 0, 2, 0, 3, 0, 1],
}


In [ ]:
def replay_solution(env_type: str, solution: list, max_steps: int = 50) -> dict:
    """Replay a pre-prepared solution through a FloPyArcade environment.

    Args:
        env_type: One of '1s-d', '2s-d', '3s-d'.
        solution: List of action indices.
        max_steps: Safety limit on number of steps.

    Returns:
        dict with 'reward', 'steps', 'final_obs'.
    """
    env = PlayEnv(envtype=env_type, ngames=min(len(solution), max_steps))
    obs = env.reset()
    total_reward = 0.0
    for i, action in enumerate(solution[:max_steps]):
        obs, reward, done, info = env.step(action)
        total_reward += reward
        if done:
            break
    return {
        'reward': total_reward,
        'steps': i + 1,
        'final_obs': obs,
    }


## 1-Step Drainage (1s-d)


In [ ]:
result = replay_solution('1s-d', SOLUTIONS['1s-d'])
print(f"Steps: {result['steps']}")
print(f"Total reward: {result['reward']:.4f}")


## 2-Step Drainage (2s-d)


In [ ]:
result = replay_solution('2s-d', SOLUTIONS['2s-d'])
print(f"Steps: {result['steps']}")
print(f"Total reward: {result['reward']:.4f}")


## 3-Step Drainage (3s-d)


In [ ]:
result = replay_solution('3s-d', SOLUTIONS['3s-d'])
print(f"Steps: {result['steps']}")
print(f"Total reward: {result['reward']:.4f}")


## Summary Table


In [ ]:
import pandas as pd

rows = []
for env_type, sol in SOLUTIONS.items():
    r = replay_solution(env_type, sol)
    rows.append({'env': env_type, 'steps': r['steps'], 'reward': round(r['reward'], 4)})

df = pd.DataFrame(rows)
print(df.to_string(index=False))


## Notes

- These solutions are **demonstration sequences**, not trained policies.
- They serve as reference baselines: a trained agent should match or exceed these rewards.
- For production training, see `flopyarcade_solvers/` and the Ray RLlib / Stable-Baselines scripts.
